# Week 21, Day 5: Integrate-and-fire neurons and the f-I curve

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w021/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Formula against simulation

Complete the Euler update in `lif_rate` (1 s of simulation with dt = 0.1 ms, returning spikes per second) and the formula in `rate_theory`, then compare them across inputs from 0 to 60 mV.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
tau, theta, tref, dt = 20.0, 15.0, 2.0, 0.1          # ms, mV, ms, ms

def lif_rate(RI, T=1000.0):
    V, refr, n = 0.0, 0.0, 0
    for _ in range(int(T / dt)):
        if refr > 0:
            refr -= dt; continue
        V = ...                                       # one Euler step of tau dV/dt = -V + RI
        if V >= theta:
            n += 1; V = 0.0; refr = tref
    return n / (T / 1000)

def rate_theory(RI):
    out = np.zeros(len(RI))
    ok = RI > theta
    out[ok] = ...                                     # spikes per second (times are in ms)
    return out

Is = np.arange(0, 61, 2.5)
sim = np.array([lif_rate(RI) for RI in Is])
print("RI = 20, 30, 60 mV:", [round(lif_rate(r), 1) for r in (20.0, 30.0, 60.0)], "Hz simulated,", rate_theory(np.array([20.0, 30.0, 60.0])).round(1), "Hz formula")
plt.plot(Is, rate_theory(Is), label="formula"); plt.plot(Is, sim, "o", label="simulation")
plt.xlabel("RI (mV)"); plt.ylabel("rate (Hz)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_silent():
    "the neuron is silent below rheobase, in both theory and simulation"
    assert lif_rate(14.0) == 0 and rate_theory(np.array([14.0]))[0] == 0

def test_match():
    "simulated rates are within 3% of the formula from 20 to 60 mV"
    for RI in (20.0, 30.0, 45.0, 60.0):
        assert abs(lif_rate(RI) - rate_theory(np.array([RI]))[0]) / rate_theory(np.array([RI]))[0] < 0.03

test_silent()
test_match()
print("All checks passed")

### Exercise: Firing below rheobase

Simulate all the input levels at once as a vector V (one entry per input), adding noise of size sigma with the update above, for 2 s, and store the rates (Hz) in `rates[sigma]` for sigma = 0, 2 and 4 mV.

In [ ]:
r_n = np.random.default_rng(1907)
Is_n = np.arange(0.0, 42.0, 2.0)
rates = {}
for sigma in (0.0, 2.0, 4.0):
    V = np.zeros(len(Is_n)); refr = np.zeros(len(Is_n)); n = np.zeros(len(Is_n))
    T = 2000.0
    for _ in range(int(T / dt)):
        free = refr <= 0
        V = np.where(free, ..., V)                    # Euler step with noise, only for neurons not refractory
        refr -= dt
        spk = V >= theta
        n += spk; V[spk] = 0.0; refr[spk] = tref
    rates[sigma] = n / (T / 1000)
print({s: rates[s][[6, 7, 10, 20]].round(1) for s in rates}, " Hz at RI = 12, 14, 20 and 40 mV")
for s in rates: plt.plot(Is_n, rates[s], "o-", label=f"sigma = {s} mV")
plt.axvline(theta, c="k", ls="--"); plt.xlabel("RI (mV)"); plt.ylabel("rate (Hz)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_below():
    "with sigma = 4 mV the neuron fires at RI = 12 mV, below rheobase, at more than 3 Hz"
    assert rates[4.0][list(Is_n).index(12.0)] > 3

def test_silent_no_noise():
    "without noise it is silent there"
    assert rates[0.0][list(Is_n).index(12.0)] == 0

def test_smooth():
    "far above rheobase noise changes the rate by less than 15%"
    i = list(Is_n).index(40.0)
    assert abs(rates[4.0][i] - rates[0.0][i]) / rates[0.0][i] < 0.15

test_below()
test_silent_no_noise()
test_smooth()
print("All checks passed")

### Exercise: Spike-frequency adaptation

Simulate 500 ms with RI = 30 mV, with b = 3 mV and tau_w = 100 ms, and without adaptation (b = 0). Record spike times in `spikes_adapt` and `spikes_plain`, and the adapting neuron's interspike intervals in `isi`.

In [ ]:
def run(b, tau_w=100.0, RI=30.0, T=500.0):
    V, w, refr, spikes = 0.0, 0.0, 0.0, []
    for i in range(int(T / dt)):
        w = w - dt / tau_w * w
        if refr > 0:
            refr -= dt; continue
        V = ...                                       # Euler step including -w
        if V >= theta:
            spikes.append(i * dt); V = 0.0; refr = tref
            w = ...
    return np.array(spikes)

spikes_plain, spikes_adapt = run(0.0), run(3.0)
isi = np.diff(spikes_adapt)
print(f"{len(spikes_plain)} spikes without adaptation, {len(spikes_adapt)} with;  first ISI {isi[0]:.1f} ms, last {isi[-1]:.1f} ms")
plt.plot(isi, "o-"); plt.xlabel("interval number"); plt.ylabel("ISI (ms)"); plt.show()

In [ ]:
import numpy as np
def test_slowing():
    "the last interspike interval is at least 1.5 times the first"
    assert isi[-1] > 1.5 * isi[0]

def test_fewer():
    "the adapting neuron fires fewer spikes in 500 ms than the plain one"
    assert len(spikes_adapt) < len(spikes_plain)

test_slowing()
test_fewer()
print("All checks passed")